In [1]:
import sys
import pandas as pd
import logging
sys.path.append('/Users/BKieft/Metabolomics/metatlas2/metatlas2')
import database_interact as dbi
import pubchem_retrieval as pcr
import load_tools as ldt
import lcmsruns_tools as lrt
import ms1_ms2_analysis as msa
import rt_align_tools as rat
import targeted_analysis as tga
import targeted_gui as tgi
import logging_config as lcf

logger = lcf.setup_logging(log_level=logging.INFO)
pd.options.display.max_colwidth = 300

In [2]:
# Required: load config file
CONFIG = ldt.load_metatlas_config("/Users/BKieft/Metabolomics/metatlas2/configs/metatlas2_config.yaml")
# Required: table with compounds to add to db. Minimally, must have columns 'inchi_key' and 'label'
COMPOUND_INPUTS = ["/Users/BKieft/Metabolomics/metatlas2/data/atlas_tables/HILICZ/HILICZ_QC_POS.tsv",
                   "/Users/BKieft/Metabolomics/metatlas2/data/atlas_tables/HILICZ/HILICZ_ISTD_POS.tsv", 
                   "/Users/BKieft/Metabolomics/metatlas2/data/atlas_tables/HILICZ/HILICZ_ISTD_NEG.tsv",
                   "/Users/BKieft/Metabolomics/metatlas2/data/atlas_tables/HILICZ/HILICZ_EMA_POS.tsv",
                   "/Users/BKieft/Metabolomics/metatlas2/data/atlas_tables/HILICZ/HILICZ_EMA_NEG.tsv"]
# Required: table of compounds; minimally, must have columns 'inchi_key', 'chromatography', 'polarity', 'rt_peak', 'mz', 'adduct'
ATLAS_INPUTS = ["/Users/BKieft/Metabolomics/metatlas2/data/atlas_tables/HILICZ/HILICZ_ISTD_POS.tsv",
                "/Users/BKieft/Metabolomics/metatlas2/data/atlas_tables/HILICZ/HILICZ_QC_POS.tsv"]
# Required: name of atlas (does not have to be unique) 
ATLAS_NAMES = ["HILIC Positive ISTD Default", "HILIC Positive QC Default"]
# Optional: description of atlas
ATLAS_DESC = [f"Default ISTD compounds for HILIC Positive", f"Default QC compounds for HILIC Positive"]
# Required: Project name
PROJECT_NAME = "20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519"
# Required: paths based on config
PROJECT_FILES_PATH = f"{CONFIG['paths']['projects_dir']}/{PROJECT_NAME}/lcmsruns"
PROJECT_DB_PATH = f"{CONFIG['paths']['projects_dir']}/{PROJECT_NAME}/{PROJECT_NAME}.duckdb"
# Required: path to id report
ANALYSIS_OUTPUT_PATH = f'/Users/BKieft/Downloads/{PROJECT_NAME}'

In [3]:
new_main_database = True
new_atlases = True
new_rt_alignment = True
new_analysis = True

In [4]:
if new_main_database is True:
    dbi.create_metatlas_database(CONFIG)
    for COMPOUNDS_INPUT in COMPOUND_INPUTS:
        compounds_df = ldt.load_compound_input(COMPOUNDS_INPUT)
        pcr.retrieve_pubchem_info(compounds_df, CONFIG)
        dbi.add_compounds_to_db(compounds_df, CONFIG, COMPOUNDS_INPUT)
    dbi.validate_database(CONFIG)

2025-09-05 16:44:35 | metatlas2.database_interact | WARNING | Overwriting existing main database (overwrite_existing=True)
2025-09-05 16:44:35 | metatlas2.database_interact | INFO | Deleted existing database at /Users/BKieft/Metabolomics/metatlas2/data/databases/metatlas.duckdb
2025-09-05 16:44:35 | metatlas2.database_interact | INFO | Main metatlas database created at /Users/BKieft/Metabolomics/metatlas2/data/databases/metatlas.duckdb
2025-09-05 16:44:35 | metatlas2.load_tools | INFO | Loaded 20 compounds from /Users/BKieft/Metabolomics/metatlas2/data/atlas_tables/HILICZ/HILICZ_QC_POS.tsv
2025-09-05 16:44:35 | metatlas2.pubchem_retrieval | INFO | Loaded global PubChem cache with 482 entries from /Users/BKieft/Metabolomics/metatlas2/data/databases/pubchem_cache/pubchem_global_cache.pkl
2025-09-05 16:44:35 | metatlas2.pubchem_retrieval | INFO | Compounds already in cache: 20
2025-09-05 16:44:35 | metatlas2.pubchem_retrieval | INFO | Compounds needing PubChem lookup: 0
2025-09-05 16:44:3

Preparing compounds:   0%|          | 0/20 [00:00<?, ?it/s]

2025-09-05 16:44:36 | metatlas2.database_interact | INFO | Batch inserting 20 compounds...
2025-09-05 16:44:36 | metatlas2.database_interact | INFO | Checking for duplicate references and batch inserting...
2025-09-05 16:44:36 | metatlas2.database_interact | INFO | Compounds added successfully!
2025-09-05 16:44:36 | metatlas2.database_interact | INFO |    Total compounds in database: 20
2025-09-05 16:44:36 | metatlas2.database_interact | INFO |    New compounds created: 20
2025-09-05 16:44:36 | metatlas2.database_interact | INFO |    Total RT/MZ references in database: 20
2025-09-05 16:44:36 | metatlas2.database_interact | INFO |    New RT/MZ references created: 20
2025-09-05 16:44:36 | metatlas2.load_tools | INFO | Loaded 65 compounds from /Users/BKieft/Metabolomics/metatlas2/data/atlas_tables/HILICZ/HILICZ_ISTD_POS.tsv
2025-09-05 16:44:36 | metatlas2.pubchem_retrieval | INFO | Loaded global PubChem cache with 482 entries from /Users/BKieft/Metabolomics/metatlas2/data/databases/pubche

Preparing compounds:   0%|          | 0/65 [00:00<?, ?it/s]

2025-09-05 16:44:36 | metatlas2.database_interact | INFO | Batch inserting 45 compounds...
2025-09-05 16:44:36 | metatlas2.database_interact | INFO | Checking for duplicate references and batch inserting...
2025-09-05 16:44:36 | metatlas2.database_interact | INFO | Compounds added successfully!
2025-09-05 16:44:36 | metatlas2.database_interact | INFO |    Total compounds in database: 65
2025-09-05 16:44:36 | metatlas2.database_interact | INFO |    New compounds created: 45
2025-09-05 16:44:36 | metatlas2.database_interact | INFO |    Compounds skipped (already existed): 20
2025-09-05 16:44:36 | metatlas2.database_interact | INFO |    Total RT/MZ references in database: 65
2025-09-05 16:44:36 | metatlas2.database_interact | INFO |    New RT/MZ references created: 45
2025-09-05 16:44:36 | metatlas2.database_interact | INFO |    RT/MZ references skipped (duplicates): 20
2025-09-05 16:44:36 | metatlas2.load_tools | INFO | Loaded 85 compounds from /Users/BKieft/Metabolomics/metatlas2/data/a

...

2025-09-05 16:44:36 | metatlas2.database_interact | INFO | Batch inserting 22 compounds...
2025-09-05 16:44:36 | metatlas2.database_interact | INFO | Checking for duplicate references and batch inserting...
2025-09-05 16:44:36 | metatlas2.database_interact | INFO | Compounds added successfully!
2025-09-05 16:44:36 | metatlas2.database_interact | INFO |    Total compounds in database: 87
2025-09-05 16:44:36 | metatlas2.database_interact | INFO |    New compounds created: 22
2025-09-05 16:44:36 | metatlas2.database_interact | INFO |    Compounds skipped (already existed): 63
2025-09-05 16:44:36 | metatlas2.database_interact | INFO |    Total RT/MZ references in database: 148
2025-09-05 16:44:36 | metatlas2.database_interact | INFO |    New RT/MZ references created: 83
2025-09-05 16:44:36 | metatlas2.database_interact | INFO |    RT/MZ references skipped (duplicates): 2
2025-09-05 16:44:36 | metatlas2.load_tools | INFO | Loaded 373 compounds from /Users/BKieft/Metabolomics/metatlas2/data/

Preparing compounds:   0%|          | 0/373 [00:00<?, ?it/s]

2025-09-05 16:44:36 | metatlas2.database_interact | INFO | Batch inserting 297 compounds...
2025-09-05 16:44:36 | metatlas2.database_interact | INFO | Checking for duplicate references and batch inserting...
2025-09-05 16:44:36 | metatlas2.database_interact | INFO | Compounds added successfully!
2025-09-05 16:44:36 | metatlas2.database_interact | INFO |    Total compounds in database: 384
2025-09-05 16:44:36 | metatlas2.database_interact | INFO |    New compounds created: 297
2025-09-05 16:44:36 | metatlas2.database_interact | INFO |    Compounds skipped (already existed): 76
2025-09-05 16:44:36 | metatlas2.database_interact | INFO |    Total RT/MZ references in database: 487
2025-09-05 16:44:36 | metatlas2.database_interact | INFO |    New RT/MZ references created: 339
2025-09-05 16:44:36 | metatlas2.database_interact | INFO |    RT/MZ references skipped (duplicates): 34
2025-09-05 16:44:36 | metatlas2.load_tools | INFO | Loaded 418 compounds from /Users/BKieft/Metabolomics/metatlas2/

Preparing compounds:   0%|          | 0/418 [00:00<?, ?it/s]

2025-09-05 16:44:36 | metatlas2.database_interact | INFO | Batch inserting 98 compounds...
2025-09-05 16:44:36 | metatlas2.database_interact | INFO | Checking for duplicate references and batch inserting...
2025-09-05 16:44:36 | metatlas2.database_interact | INFO | Compounds added successfully!
2025-09-05 16:44:36 | metatlas2.database_interact | INFO |    Total compounds in database: 482
2025-09-05 16:44:36 | metatlas2.database_interact | INFO |    New compounds created: 98
2025-09-05 16:44:36 | metatlas2.database_interact | INFO |    Compounds skipped (already existed): 320
2025-09-05 16:44:36 | metatlas2.database_interact | INFO |    Total RT/MZ references in database: 860
2025-09-05 16:44:36 | metatlas2.database_interact | INFO |    New RT/MZ references created: 373
2025-09-05 16:44:36 | metatlas2.database_interact | INFO |    RT/MZ references skipped (duplicates): 45
2025-09-05 16:44:36 | metatlas2.database_interact | INFO | Database Validation:
2025-09-05 16:44:36 | metatlas2.data

In [5]:
if new_atlases is True:
    new_atlas_info = ()
    for ATLAS_INPUT, ATLAS_NAME, ATLAS_DESC in zip(ATLAS_INPUTS, ATLAS_NAMES, ATLAS_DESC):
        atlas_compounds_df = ldt.load_atlas_input(ATLAS_INPUT)
        atlas_uid, atlas_name = dbi.create_atlas_from_compounds(atlas_compounds_df, ATLAS_NAME, ATLAS_DESC, CONFIG)
        new_atlas_info += ((atlas_uid, atlas_name),)
    dbi.validate_database(CONFIG)
    QC_ATLAS_UID = next(uid for uid, name in new_atlas_info if "QC" in name)
    TARGET_ATLAS_UID = next(uid for uid, name in new_atlas_info if "QC" not in name)
else:
    QC_ATLAS_UID = "atl-raw-7906f4bf395d4ed791f3763ed10f61c8"
    TARGET_ATLAS_UID = "atl-raw-2e24a9969a6a4758844b99162e9e0a03"
    logger.info(f"Skipping new atlas creation and using QC Atlas: {QC_ATLAS_UID} and TARGET Atlas: {TARGET_ATLAS_UID}")

2025-09-05 16:44:36 | metatlas2.load_tools | INFO | Loaded 65 atlas entries from /Users/BKieft/Metabolomics/metatlas2/data/atlas_tables/HILICZ/HILICZ_ISTD_POS.tsv
2025-09-05 16:44:37 | metatlas2.database_interact | INFO | Created atlas 'HILIC Positive ISTD Default' with UID: atl-raw-16569b581a7c44c6940a1b7084b7fc8a
2025-09-05 16:44:37 | metatlas2.database_interact | INFO |   Added 65 compound associations
2025-09-05 16:44:37 | metatlas2.load_tools | INFO | Loaded 20 atlas entries from /Users/BKieft/Metabolomics/metatlas2/data/atlas_tables/HILICZ/HILICZ_QC_POS.tsv
2025-09-05 16:44:37 | metatlas2.database_interact | INFO | Created atlas 'HILIC Positive QC Default' with UID: atl-raw-2738e845d5a94b2c9a980e57df1bf582
2025-09-05 16:44:37 | metatlas2.database_interact | INFO |   Added 20 compound associations
2025-09-05 16:44:37 | metatlas2.database_interact | INFO | Database Validation:
2025-09-05 16:44:37 | metatlas2.database_interact | INFO |    Compounds: 482
2025-09-05 16:44:37 | metatla

In [6]:
if new_rt_alignment is True:
    dbi.create_project_database(PROJECT_DB_PATH, CONFIG)
    lcmsrun_files = dbi.save_lcmsruns_to_db(PROJECT_DB_PATH, PROJECT_NAME, PROJECT_FILES_PATH, CONFIG)
    matches_df, matching_stats = msa.extract_and_match_qc_compounds(PROJECT_DB_PATH, QC_ATLAS_UID, CONFIG)
    best_model, model_results, model_stats = rat.build_rt_alignment_model(matches_df, CONFIG)
    ANALYSIS_ATLAS_UID = rat.apply_rt_correction_to_target(PROJECT_DB_PATH, TARGET_ATLAS_UID, CONFIG, best_model, lcmsrun_files, model_results)
else:
    ANALYSIS_ATLAS_UID = "atl-rta-0c89e582233e481ca4f1920611f6007f"
    logger.info(f"Skipping new RT alignment and using {ANALYSIS_ATLAS_UID}")

2025-09-05 16:44:37 | metatlas2.database_interact | INFO | Overwrite is set to True; Deleted existing database at /Users/BKieft/Metabolomics/metatlas2/data/test_data/projects/20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519/20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519.duckdb and proceeding...
2025-09-05 16:44:37 | metatlas2.database_interact | INFO | Project database created at /Users/BKieft/Metabolomics/metatlas2/data/test_data/projects/20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519/20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519.duckdb
2025-09-05 16:44:37 | metatlas2.lcmsrun_tools | INFO | Found 8 .h5 files in /Users/BKieft/Metabolomics/metatlas2/data/test_data/projects/20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519/lcmsruns


Categorizing files:   0%|          | 0/8 [00:00<?, ?it/s]

2025-09-05 16:44:37 | metatlas2.database_interact | INFO | Saved 8 LCMS runs to database:
2025-09-05 16:44:37 | metatlas2.database_interact | INFO | Chromatography: HILIC
2025-09-05 16:44:37 | metatlas2.database_interact | INFO |   Polarity: FPS
2025-09-05 16:44:37 | metatlas2.database_interact | INFO |     qc: 2 files
2025-09-05 16:44:37 | metatlas2.database_interact | INFO |     istd: 1 files
2025-09-05 16:44:37 | metatlas2.database_interact | INFO |     experimental: 1 files
2025-09-05 16:44:37 | metatlas2.database_interact | INFO |   Polarity: positive
2025-09-05 16:44:37 | metatlas2.database_interact | INFO |     experimental: 3 files
2025-09-05 16:44:37 | metatlas2.database_interact | INFO |     exctrl: 1 files
2025-09-05 16:44:37 | metatlas2.ms1_ms2_analysis | INFO | Loading QC files and atlas compounds from databases...
2025-09-05 16:44:37 | metatlas2.database_interact | INFO | Retrieved 20 compounds from main database for atlas: HILIC Positive QC Default (atl-raw-2738e845d5a94

Extracting MS1 data:   0%|          | 0/2 [00:00<?, ?it/s]

2025-09-05 16:44:37 | metatlas2.ms1_ms2_analysis | INFO |   Extracted 287578 peaks from 20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519_FPS_MS1_2_Winter-RWi_3_Rg70to1050-CE102040norm-rhizo-QC_Run58.h5 (ms1_pos)
2025-09-05 16:44:37 | metatlas2.ms1_ms2_analysis | INFO |   Extracted 124042 peaks from 20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519_FPS_MS1_2_Winter-RWi_3_Rg70to1050-CE102040norm-rhizo-QC_Run58.h5 (ms1_neg)
2025-09-05 16:44:37 | metatlas2.ms1_ms2_analysis | INFO |   Extracted 274055 peaks from 20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519_FPS_MS1_33_Summer-VSu_6_Rg70to1050-CE102040norm-veg-core-QC_Run107.h5 (ms1_pos)
2025-09-05 16:44:37 | metatlas2.ms1_ms2_analysis | INFO |   Extracted 118235 peaks from 20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519_FPS_MS1_33_Summer-VSu_6_Rg70to1050-CE102040norm-veg-core-QC_Run107.h5 (ms1_neg)
2025-09-05 16:44:37 | metatlas2.ms1_ms2_analysis | INFO | Total MS1 peaks extr

Matching compounds:   0%|          | 0/20 [00:00<?, ?it/s]

2025-09-05 16:44:40 | metatlas2.ms1_ms2_analysis | INFO | Processed 20 compounds, 20 had matches
2025-09-05 16:44:40 | metatlas2.ms1_ms2_analysis | INFO | Matching completed:
2025-09-05 16:44:40 | metatlas2.ms1_ms2_analysis | INFO |   Compounds with matches: 20/20
2025-09-05 16:44:40 | metatlas2.ms1_ms2_analysis | INFO |   Total compound-file matches: 40
2025-09-05 16:44:40 | metatlas2.ms1_ms2_analysis | INFO |   Mean m/z error: -0.21 ± 0.51 ppm
2025-09-05 16:44:40 | metatlas2.ms1_ms2_analysis | INFO |   Mean RT difference: 0.028 ± 0.119 min
2025-09-05 16:44:41 | metatlas2.rt_align_tools | INFO | Building RT alignment model...
2025-09-05 16:44:41 | metatlas2.rt_align_tools | INFO | Filtered out 2 matches by InChI Key.
2025-09-05 16:44:41 | metatlas2.rt_align_tools | INFO | RT Statistics Summary:
2025-09-05 16:44:41 | metatlas2.rt_align_tools | INFO |   Atlas RT range: 1.09 - 17.01 min
2025-09-05 16:44:41 | metatlas2.rt_align_tools | INFO |   Observed RT range (median): 1.25 - 17.14 min

,compound_name,inchi_key,ref_rt_peak,exp_rt_median,rt_diff_median,observation_count,exp_rt_std
0,"valine (U - 13C, 15N)",KZSNJWFQEVHDMF-XAFSXMPTSA-N,11.1160,11.014900,-0.1011,2,0.0159
1,"alanine (U - 13C, 15N)",QNAYBMKLOCPYGJ-UVYXLFMMSA-N,13.4051,13.358000,-0.0471,2,0.0095
2,hypoxanthine (U - 15N),FDGQSTZJBFJUBT-NNZQUYKOSA-N,3.1030,3.149900,0.0469,2,0.0215
3,guanine (U - 15N),UYTPUPDQBNUYGX-CIKZIQIKSA-N,6.2654,6.210600,-0.0548,2,0.0090
4,"lysine (U - 13C, 15N)",KDXKERNSBIXSRK-JMKXWGMHSA-N,17.0113,17.136400,0.1251,2,0.0051
5,"glutamic acid (U - 13C, 15N)",WHUUTDBJXJRKMK-UFLWJPECSA-N,15.9354,15.966100,0.0307,2,0.0090
6,"tyrosine (U - 13C, 15N)",OUYCCCASQSFEME-CMLFETTRSA-N,11.8552,11.751900,-0.1032,2,0.0209
7,"asparagine (U - 13C, 15N)",DCXYFEDJOCDNAF-FLEDYEEXSA-N,14.3681,14.318600,-0.0495,2,0.0003
8,"tryptophan (U - 13C, 15N)",QIVBCDIJIAJPQS-HNEHNLBWSA-N,10.1566,10.180700,0.0240,2,0.0109
9,"methionine (U - 13C, 15N)",FFEARJCKVFRZRR-XAFSXMPTSA-N,10.4410,10.409400,-0.0315,2,0.0075


2025-09-05 16:44:41 | metatlas2.rt_align_tools | INFO | Cloning target atlas and applying RT correction...
2025-09-05 16:44:41 | metatlas2.database_interact | INFO | Retrieved atlas metadata for: HILIC Positive ISTD Default
2025-09-05 16:44:41 | metatlas2.database_interact | INFO | Retrieved 65 compounds from main database for atlas: HILIC Positive ISTD Default (atl-raw-16569b581a7c44c6940a1b7084b7fc8a)
2025-09-05 16:44:41 | metatlas2.database_interact | INFO | Creating RT-corrected atlas in project database...
2025-09-05 16:44:41 | metatlas2.database_interact | INFO | Created RT-corrected atlas: atl-rta-8e513b6a597a480484c93f054d8e3e6c name: HILIC Positive ISTD Default (RT Corrected)
2025-09-05 16:44:41 | metatlas2.database_interact | INFO |   Atlas name: HILIC Positive ISTD Default (RT Corrected)
2025-09-05 16:44:41 | metatlas2.database_interact | INFO |   Corrected 65 compounds
2025-09-05 16:44:41 | metatlas2.database_interact | INFO |   Mean RT shift: 0.0026 min
2025-09-05 16:44:41

In [7]:
if new_analysis is True:
    # Run targeted analysis workflow - now returns ProjectAnalysis object
    atlas_dataframe, project_analysis = tga.run_targeted_analysis_workflow(
        PROJECT_DB_PATH, ANALYSIS_ATLAS_UID, CONFIG
    )

    # Create GUI
    gui = tgi.create_gui(project_analysis, CONFIG, PROJECT_DB_PATH)
    gui._project_analysis = project_analysis  # Store ProjectAnalysis object for new workflow
    display(gui)

2025-09-05 16:44:41 | metatlas2.targeted_analysis | INFO | Setting up targeted analysis database...
2025-09-05 16:44:41 | metatlas2.targeted_analysis | INFO | Running fresh targeted analysis...
2025-09-05 16:44:41 | metatlas2.targeted_analysis | INFO | Loading target atlas...
2025-09-05 16:44:41 | metatlas2.database_interact | INFO | Retrieved 65 compounds from project database for atlas: HILIC Positive ISTD Default (RT Corrected) (atl-rta-8e513b6a597a480484c93f054d8e3e6c)
2025-09-05 16:44:41 | metatlas2.database_interact | INFO | RT-corrected atlas detected with experimental data
2025-09-05 16:44:41 | metatlas2.database_interact | INFO | Retrieved 65 compounds with metadata using external database attachment
2025-09-05 16:44:41 | metatlas2.targeted_analysis | INFO | Created Atlas dataframe with 65 compounds
2025-09-05 16:44:41 | metatlas2.simple_cache | INFO | Progress checkpoint saved: initialized at 20250905_164441
2025-09-05 16:44:41 | metatlas2.targeted_analysis | INFO | Loading e

Processing files in parallel:   0%|          | 0/6 [00:00<?, ?it/s]

2025-09-05 16:44:45 | metatlas2.ms1_ms2_analysis | INFO |   Completed 1/6: 20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519_FPS_MS1_22_Fall-RFa_7_Rg70to1050-CE205060norm-rhizo-InjBL-MeOH_Run123.h5 - 36 compounds
2025-09-05 16:44:45 | metatlas2.ms1_ms2_analysis | INFO |   Completed 2/6: 20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519_FPS_MS1_28_Spring-VSp_2_Rg70to1050-CE102040norm-veg-core-ISTD_Run31.h5 - 49 compounds
2025-09-05 16:44:46 | metatlas2.ms1_ms2_analysis | INFO |   Completed 3/6: 20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519_POS_MS2_36_Fall-VFa_7_Rg70to1050-CE102040norm-veg-core-S1_Run136.h5 - 51 compounds
2025-09-05 16:44:46 | metatlas2.ms1_ms2_analysis | INFO |   Completed 4/6: 20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519_POS_MS2_48_Fall-BFa_7_Rg70to1050-CE102040norm-nonveg-core-S1_Run127.h5 - 56 compounds
2025-09-05 16:44:46 | metatlas2.ms1_ms2_analysis | INFO |   Completed 5/6: 20250707_JGI_MS_510172

In [8]:
if 'project_analysis' in locals():
    sample_inchi_key = 'AGPKZVBTJJNPAG-WHFBIAKZSA-N'
    sample_compound = project_analysis.compounds[sample_inchi_key]
    print(sample_compound)
    print(f"Sample compound: {sample_compound.compound_name}")
    print(f"Original RT: {sample_compound.original_rt_peak}")
    print(f"Current RT: {sample_compound.rt_peak}")
    print(f"Is modified: {sample_compound.is_rt_modified or sample_compound.is_annotation_modified}")
    print(f"EIC files: {len(sample_compound.eic_data_files)}")
    print(f"MS2 files: {len(sample_compound.ms2_data_files.get('files', {}))}")

CompoundData(compound_uid='cmp-f2a0a3bd36f74c3d9cd34c78b9e43249', inchi_key='AGPKZVBTJJNPAG-WHFBIAKZSA-N', compound_name='isoleucine (unlabeled)', formula='', mz=132.10189819335938, adduct='[M+H]+', polarity='positive', chromatography='HILICZ', mz_tolerance=5.0, original_rt_peak=9.636879920959473, original_rt_min=9.33687973022461, original_rt_max=9.936880111694336, rt_peak=9.636879920959473, rt_min=9.33687973022461, rt_max=9.936880111694336, ms1_notes='keep', ms2_notes='no selection', analyst_notes='', identification_notes='', best_eic_file='20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519_POS_MS2_9_Spring-RSp_6_Rg70to1050-CE102040norm-rhizo-S1_Run99.h5', best_eic_rt=9.498273849487305, best_eic_mz=132.10182189941406, best_eic_intensity=5603004.0, best_eic_ppm_error=0.11550777332662156, best_eic_rt_error=9.498273849487305, avg_eic_rt=9.508100668589273, avg_eic_intensity=1151822.8841145833, avg_eic_mz=132.10183461507162, best_ms2_file='20250707_JGI_MS_510172_SedRhizVegOut

In [9]:
if new_analysis is True:
    post_analysis_atlas_uid, targeted_analysis_uid, comprehensive_report = tga.run_post_analysis_workflow_v2(
        project_db_path=PROJECT_DB_PATH,
        analysis_atlas_uid=ANALYSIS_ATLAS_UID,
        project_analysis=gui._project_analysis,  # Use ProjectAnalysis object
        atlas_dataframe=atlas_dataframe,
        project_name=PROJECT_NAME,
        config=CONFIG,
        analysis_output_path=ANALYSIS_OUTPUT_PATH
    )
    
    logger.info("Class-based workflow completed successfully!")

# Print results
logger.info(f"Analysis saved with UID: {targeted_analysis_uid}")
logger.info(f"Post-analysis atlas UID: {post_analysis_atlas_uid}")
logger.info(f"Report saved to: {ANALYSIS_OUTPUT_PATH}")

2025-09-05 16:44:48 | metatlas2.targeted_analysis | INFO | Starting class-based post-analysis workflow...
2025-09-05 16:44:48 | metatlas2.targeted_analysis | INFO | Creating post-analysis atlas...
2025-09-05 16:44:48 | metatlas2.targeted_analysis | INFO | No compounds were modified during analysis
2025-09-05 16:44:48 | metatlas2.targeted_analysis | INFO | Created post-analysis atlas: atl-rta-8e513b6a597a480484c93f054d8e3e6c
2025-09-05 16:44:48 | metatlas2.targeted_analysis | INFO | Saving targeted analysis results...
2025-09-05 16:44:48 | metatlas2.database_interact | INFO | Saving targeted analysis results for project '20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519' using class-based approach...
2025-09-05 16:44:48 | metatlas2.database_interact | INFO | Validation successful for analysis tga-5d92f8ede87b4a52b9dda780e8cc9829:
2025-09-05 16:44:48 | metatlas2.database_interact | INFO |   Total records: 65
2025-09-05 16:44:48 | metatlas2.database_interact | INFO |   Reco

Processing compounds:   0%|          | 0/65 [00:00<?, ?it/s]

2025-09-05 16:44:48 | metatlas2.database_interact | ERROR | Error saving Excel file: [Errno 21] Is a directory: '/Users/BKieft/Downloads/20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519'
2025-09-05 16:44:48 | metatlas2.database_interact | ERROR | Error saving Excel file: [Errno 21] Is a directory: '/Users/BKieft/Downloads/20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519'
2025-09-05 16:44:48 | metatlas2.targeted_analysis | INFO | Generated comprehensive report and saved to /Users/BKieft/Downloads/20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519
2025-09-05 16:44:48 | metatlas2.targeted_analysis | INFO | Class-based post-analysis workflow completed successfully
2025-09-05 16:44:48 | metatlas2 | INFO | Class-based workflow completed successfully!
2025-09-05 16:44:48 | metatlas2 | INFO | Analysis saved with UID: tga-5d92f8ede87b4a52b9dda780e8cc9829
2025-09-05 16:44:48 | metatlas2 | INFO | Post-analysis atlas UID: atl-rta-8e513b6a597a480484c93f

In [10]:
dbi.validate_database(CONFIG, PROJECT_DB_PATH)

2025-09-05 16:44:48 | metatlas2.database_interact | INFO | Database Validation:
2025-09-05 16:44:48 | metatlas2.database_interact | INFO |    Atlases: 1
2025-09-05 16:44:48 | metatlas2.database_interact | INFO |    Targeted analyses: 1
2025-09-05 16:44:48 | metatlas2.database_interact | INFO |    RT alignment models: 1
2025-09-05 16:44:48 | metatlas2.database_interact | INFO |    Experimental RT/MZ entries: 65
2025-09-05 16:44:48 | metatlas2.database_interact | INFO |    Available atlases:
2025-09-05 16:44:48 | metatlas2.database_interact | INFO |       atl-rta-8e513b6a597a480484c93f054d8e3e6c
2025-09-05 16:44:48 | metatlas2.database_interact | INFO |    Targeted analysis entries:
2025-09-05 16:44:48 | metatlas2.database_interact | INFO |       tga-5d92f8ede87b4a52b9dda780e8cc9829 (20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519) - Atlas: atl-rta-8e513b6a597a480484c93f054d8e3e6c - 65 compounds


BinderException: Binder Error: Referenced column "last_modified" not found in FROM clause!
Candidate bindings: "atlas_uid", "rt_alignment_uid", "model_type", "polynomial_degree", "created_date"

LINE 2: ..., atlas_uid, model_type, polynomial_degree, r_squared, rmse, last_modified
                                                                        ^